---
last_verified: 2026-09-10
tool_version: n/a
sources:
  - https://pre-commit.com
---

# Pre-commit Hook Debugging: Tracing pass_filenames, pass_args, and Hook-Stage Flow

This notebook walks through how pre-commit passes filenames and arguments to hooks, how hook stages control execution timing, and how to trace the full flow with prints and logs when debugging custom hooks.

## 1. How pass_filenames Works

By default, `pass_filenames: true` — pre-commit appends the list of staged files (matching the hook's `files`/`types` patterns) as positional arguments to the hook entry point.

When `pass_filenames: false`, the hook receives **only** the `args` from the config, with no file list. This is useful for hooks that manage their own file discovery (e.g., a hook that scans the entire repo).

In [ ]:
# Example: .pre-commit-config.yaml snippet demonstrating pass_filenames
config_with_filenames = """
repos:
  - repo: local
    hooks:
      - id: trace-filenames
        name: Trace filenames passed to hook
        entry: python -c "import sys; print('Received files:', sys.argv[1:])"
        language: system
        pass_filenames: true
        always_run: true
"""
print(config_with_filenames)

When this hook runs, the output will show the staged files as arguments:

```
Received files: ['src/main.py', 'tests/test_main.py']
```

With `pass_filenames: false`, the output would be:

```
Received files: []
```

## 2. How args Flow to Hooks

The `args` list in `.pre-commit-config.yaml` is prepended to the file list when `pass_filenames: true`. The final command becomes:

```
<entry> <args[0]> <args[1]> ... <file1> <file2> ...
```

This means your hook script receives args first, then filenames.

In [ ]:
# Example: args + pass_filenames combined
config_with_args = """
repos:
  - repo: local
    hooks:
      - id: trace-args-and-files
        name: Trace args and filenames
        entry: python -c "import sys; print('Args:', sys.argv[1:3]); print('Files:', sys.argv[3:])"
        language: system
        args: ['--verbose', '--strict']
        pass_filenames: true
        always_run: true
"""
print(config_with_args)

Running this produces:

```
Args: ['--verbose', '--strict']
Files: ['src/main.py']
```

Key insight: `args` are static (defined in config), filenames are dynamic (determined at runtime by which files match the `files`/`types` patterns).

## 3. Hook Stages: When Hooks Fire

The `stages` property controls which git event triggers the hook. Default is all stages if unset. The supported stages map to git hooks:

| Stage | Git Hook | When It Fires |
|-------|----------|---------------|
| `pre-commit` | `.git/hooks/pre-commit` | Before commit is finalized |
| `pre-push` | `.git/hooks/pre-push` | On `git push` |
| `commit-msg` | `.git/hooks/commit-msg` | After commit message is written |
| `pre-merge-commit` | `.git/hooks/pre-merge-commit` | After merge succeeds, before merge commit |
| `manual` | (none) | Only via `pre-commit run --hook-stage manual` |

New in pre-commit 3.2.0: stage values match hook names directly (previously `commit` mapped to `pre-commit`).

In [ ]:
# Example: hooks confined to specific stages
config_stages = """
repos:
  - repo: local
    hooks:
      - id: lint-on-commit
        name: Lint on commit
        entry: python -m ruff check
        language: system
        stages: [pre-commit]
        pass_filenames: true

      - id: typecheck-on-push
        name: Typecheck on push
        entry: python -m mypy src/
        language: system
        stages: [pre-push]
        pass_filenames: false

      - id: validate-commit-msg
        name: Validate commit message
        entry: python -c "import sys; msg=open(sys.argv[1]).read(); assert len(msg.splitlines()[0]) <= 72"
        language: system
        stages: [commit-msg]
        pass_filenames: false

      - id: manual-audit
        name: Manual security audit
        entry: python -m bandit -r src/
        language: system
        stages: [manual]
        pass_filenames: false
"""
print(config_stages)

### Debugging Stage Flow

To trace which stage is active during a run, inspect the `PRE_COMMIT` environment variable and use `--hook-stage` on the command line:

```bash
# Run a hook at a specific stage for debugging
pre-commit run manual-audit --hook-stage manual

# Force-run all hooks at the pre-commit stage
pre-commit run --all-files --hook-stage pre-commit
```

## 4. Tracing with Prints and Logs

When a hook fails, pre-commit shows the exit code and any stdout/stderr. For deeper tracing:

- Use `--verbose` to force output even on pass
- Use `--show-diff-on-failure` to run `git diff` after a hook fails
- Set `verbose: true` in the hook config to always print output
- Set `log_file: <path>` in the hook config to capture output to a file

In [ ]:
# Example: debugging hook with verbose output and log file
config_debug = """
repos:
  - repo: local
    hooks:
      - id: debug-trace
        name: Debug trace hook
        entry: python scripts/debug_hook.py
        language: system
        pass_filenames: true
        verbose: true
        log_file: .pre-commit-debug.log
        always_run: true
"""
print(config_debug)

### Example debug_hook.py

A hook script that logs its invocation context for debugging:

In [ ]:
# scripts/debug_hook.py
import os
import sys
from datetime import datetime

def main():
    """Debug hook that traces all pre-commit context."""
    log_lines = []
    log_lines.append(f"=== Hook invocation at {datetime.now().isoformat()} ===")
    log_lines.append(f"PRE_COMMIT env: {os.environ.get('PRE_COMMIT', 'not set')}")
    log_lines.append(f"sys.argv: {sys.argv}")
    log_lines.append(f"Files passed ({len(sys.argv) - 1}): {sys.argv[1:]}")

    # Trace each file
    for filepath in sys.argv[1:]:
        if os.path.exists(filepath):
            size = os.path.getsize(filepath)
            log_lines.append(f"  {filepath}: {size} bytes")
        else:
            log_lines.append(f"  {filepath}: NOT FOUND")

    output = "\n".join(log_lines)
    print(output)
    return 0

if __name__ == "__main__":
    sys.exit(main())

Running `pre-commit run debug-trace --all-files --verbose` produces output like:

```
Debug trace hook.......................................................Passed
- hook id: debug-trace
- duration: 0.02s

=== Hook invocation at 2026-09-10T14:30:00 ===
PRE_COMMIT env: 1
sys.argv: ['scripts/debug_hook.py', 'src/main.py', 'tests/test_main.py']
Files passed (2): ['src/main.py', 'tests/test_main.py']
  src/main.py: 1234 bytes
  tests/test_main.py: 567 bytes
```

## 5. The PRE_COMMIT Environment Variable

Since pre-commit 2.5.0, the `PRE_COMMIT=1` environment variable is set during hook execution. Your hook can check this to distinguish between being run by pre-commit vs. being run directly.

In [ ]:
# Example: hook that behaves differently under pre-commit vs direct invocation
import os
import sys

def main():
    in_pre_commit = os.environ.get("PRE_COMMIT") == "1"

    if in_pre_commit:
        print("Running under pre-commit — strict mode")
        # Stricter checks when run as a hook
    else:
        print("Running directly — lenient mode")
        # Relaxed checks for manual invocation

    files = sys.argv[1:] if in_pre_commit else []
    print(f"Files to check: {files}")
    return 0

if __name__ == "__main__":
    sys.exit(main())

## 6. Debugging with pre-commit try-repo

`pre-commit try-repo` lets you test hooks from a local or remote repo without adding them to your config. This is the fastest way to iterate on a custom hook during development.

In [ ]:
# Example: try-repo workflow for debugging a custom hook
try_repo_workflow = """
# In the hook repo directory:
cd ~/my-hook-repo
git add .
git commit -m 'wip: debug tracing'

# In the target project directory:
cd ~/my-project
pre-commit try-repo ~/my-hook-repo my-hook-id --verbose --all-files

# Output shows the generated config and hook output:
# ================================================================================
# Using config:
# ================================================================================
# repos:
# -   repo: /home/user/my-hook-repo
#     rev: 84f01ac09fcd8610824f9626a590b83cfae9bcbd
#     hooks:
#     -   id: my-hook-id
# ================================================================================
# [INFO] Initializing environment for /home/user/my-hook-repo.
# My Hook...............................................................Passed
# - hook id: my-hook-id
# - duration: 0.02s
"""
print(try_repo_workflow)

## 7. Tracing Hook-Stage Flow End-to-End

Putting it all together: here's a complete debugging setup that traces the full lifecycle of a hook across stages.

In [ ]:
# Complete debugging config with stage tracing
full_debug_config = """
repos:
  - repo: local
    hooks:
      - id: stage-tracer
        name: Trace hook stage and args
        entry: python scripts/trace_stage.py
        language: system
        stages: [pre-commit, pre-push, commit-msg, manual]
        pass_filenames: true
        args: ['--trace']
        verbose: true
        log_file: .pre-commit-trace.log
"""
print(full_debug_config)

In [ ]:
# scripts/trace_stage.py
import os
import sys
from datetime import datetime

def main():
    args = sys.argv[1:]
    trace_mode = "--trace" in args
    files = [a for a in args if not a.startswith("--")]

    # Detect stage from environment or hook name
    hook_id = os.environ.get("PRE_COMMIT_HOOK_ID", "unknown")
    pre_commit = os.environ.get("PRE_COMMIT") == "1"

    lines = [
        f"[{datetime.now().isoformat()}] Hook '{hook_id}' invoked",
        f"  PRE_COMMIT={pre_commit}",
        f"  args (non-file): {[a for a in args if a.startswith('--')}]}",
        f"  files: {files}",
    ]

    for f in files:
        if os.path.isfile(f):
            with open(f) as fh:
                first_line = fh.readline().strip()
            lines.append(f"  {f}: first line = '{first_line}'")

    output = "\n".join(lines)
    print(output)

    # Write to trace log if --trace
    if trace_mode:
        with open(".pre-commit-trace.log", "a") as logf:
            logf.write(output + "\n\n")

    return 0

if __name__ == "__main__":
    sys.exit(main())

## 8. Common Debugging Scenarios

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Hook runs but file list is empty | `pass_filenames: false` or no files match `files`/`types` | Check `files` pattern; use `--all-files` to verify |
| Hook never runs | Wrong `stages` setting | Verify stage matches the git event; use `manual` stage for on-demand |
| Args not reaching script | Positional args mixed with file args | Check `sys.argv` order: args first, then files |
| Hook passes locally but fails in CI | Missing env or different working directory | Use `verbose: true` and `log_file` to capture CI output |
| Slow first run | Environment installation | Use `pre-commit install-hooks` to pre-install all environments |

## Summary

- **pass_filenames** (default `true`): controls whether staged files are appended as positional args
- **args**: static arguments prepended before the file list in `sys.argv`
- **stages**: which git event triggers the hook (`pre-commit`, `pre-push`, `commit-msg`, `manual`, etc.)
- **PRE_COMMIT=1**: environment variable set during hook execution (since v2.5.0)
- **Debugging tools**: `--verbose`, `--show-diff-on-failure`, `log_file`, `pre-commit try-repo`, `pre-commit run --hook-stage manual`
- **Trace pattern**: write a hook that logs `sys.argv`, env vars, and file metadata to a log file for post-mortem analysis